# (Figure) Plot the analytic spectrum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Choose nodes with golden ratio spacing between frequencies.
phi     = 1.618
n_nodes = 11
frange  = 2 * (phi ** np.arange(0, n_nodes, 1))

# Get the autoregressive coefficients.
Fs = 1000
r  = 0.99999
w_t_minus_1 = 2 * r * np.cos(2 * np.pi * frange / Fs)
w_t_minus_2 = (-r**2) * np.ones_like(w_t_minus_1)

# Plot the analytic spectra.
plt.figure(figsize=(3, 2.75))

N  = 1000000
dt = 1/Fs
j  = np.arange(0, N//2 + 1)
T  = N * dt
sigma = 1
freq_vector = j / T
for k in np.arange(len(w_t_minus_1)):
    denom = (
        1 
        + w_t_minus_1[k]**2 
        + w_t_minus_2[k]**2 
        - 2 * w_t_minus_1[k] * np.cos(2 * np.pi * j / N)
        - 2 * w_t_minus_2[k] * np.cos(2 * np.pi * 2 * j / N)
        + 2 * w_t_minus_1[k] * w_t_minus_2[k] * np.cos(2 * np.pi * j / N)
    )
    S = sigma**2 * dt / denom
    plt.loglog(freq_vector, S, linewidth=2)
    plt.xlim([1,500])
plt.xlabel('Frequency [Hz]')
plt.ylabel('Analytic Spectrum [a.u.]')
plt.xticks(frange, [f'{f:.1f}' for f in frange], rotation=90)
plt.minorticks_off()
plt.tick_params(axis='x', which='minor', bottom=False, top=False)
plt.tick_params(axis='y', which='both', left=False, right=False, labelleft=False)
plt.grid(axis='x', linestyle='dashed')

plt.savefig(
    "./ps/Fig1_A_Analytic_Spectra.pdf",
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.01
)
plt.show()

# (Figure) Plot example node dynamics

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
from RRN_functions import RRNModel

# Simulate golden-ratio RRN with zero input, zero recurrence, and additive noise.
Fs = 1000
duration_s = 10.0
T = int(Fs * duration_s)

model = RRNModel(Fs=Fs,rhythm_configuration='golden',no_recurrence=True,sigma=1,verbose=False,)

# Reproducible simulation
torch.manual_seed(0)

# Simulate the model.
K     = model.K
x_tm1 = torch.zeros(K, dtype=torch.float32)
x_tm2 = torch.zeros(K, dtype=torch.float32)
X     = np.zeros((T, K), dtype=np.float32)

for t in range(T):
    hist_drive = model.w1 * x_tm1 + model.w2 * x_tm2
    noise      = torch.randn(K, dtype=torch.float32) * model.sigma
    x_t        = hist_drive + noise  # zero input + zero recurrence
    X[t, :]    = x_t.numpy()
    x_tm2      = x_tm1
    x_tm1      = x_t

time_s = np.arange(T) / Fs

frange = np.asarray(model.frange)

# Plot it.
t0 = 2.0
window_s = 5
mask = (time_s >= t0) & (time_s <= t0 + window_s)
t_plot = time_s[mask]
X_plot = X[mask, :]

fig, ax = plt.subplots(figsize=(5, 3), dpi=150)

colors = plt.cm.tab10(np.linspace(0, 1, K))

# Horizontal dashed baselines at each resonant frequency
for f in frange:
    ax.axhline(f, color='0.65', ls='--', lw=0.7, zorder=0)

# Scale each trace independently, then offset by its resonant frequency
for k in range(K):
    xk = X_plot[:, k]
    rms = np.std(xk)
    scale = 1.0 if rms < 1e-12 else 0.08 * frange[k] / rms
    yk = frange[k] + scale * xk
    yk = np.maximum(yk, 1e-6)  # keep positive for log axis
    ax.plot(t_plot, yk, lw=1.2, color=colors[k], alpha=0.9)

# Axis styling
ax.set_xlim(t_plot[0], t_plot[-1])
ax.set_yscale('log')
ax.set_ylim(frange[0] / 1.6, frange[-1] * 1.25)
ax.set_ylabel('Node resonant frequency [Hz]')

# Put y-ticks at the resonant frequencies, formatted to 1 decimal place
ax.set_yticks(frange)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))

# Remove normal x-axis ticks/labels
ax.set_xticks([])

# Minimal spines, like the example
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_linewidth(1.2)
ax.spines['left'].set_color('0.25')
ax.minorticks_off()
ax.tick_params(axis='y', which='minor', left=False, right=False)
ax.tick_params(axis='y', which='major', left=True, right=False, length=0, width=0.8, labelsize=9, labelcolor='k')

# Add a 100 ms scale bar
bar_len_s = 0.10
x_bar0 = t_plot[0] + 0.05 * (t_plot[-1] - t_plot[0])
x_bar1 = x_bar0 + bar_len_s
y_bar = frange[0] / 1.35
ax.plot([x_bar0, x_bar1], [y_bar, y_bar], color='0.2', lw=1.2, solid_capstyle='butt')
ax.text((x_bar0 + x_bar1) / 2, y_bar / 1.12, '100 ms', ha='center', va='top', color='0.2', fontsize=9)

plt.tight_layout()

plt.savefig(
    "./ps/Fig1_B_Example_Traces.pdf",
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.01
)

plt.show()

# RRN (slow): Parameter grid search for each architecture.

In [ ]:
import os, itertools

from load_MNIST    import make_timeseries_loaders
from Utilities     import append_row, run_one
from RRN_functions import RRNModel, rrn_prepare_batch

# ---------------- Settings ----------------
EPOCHS      = 30
SEEDS       = [123, 456, 789]
NUM_WORKERS = 0
PRINT_STATS = False

OUT_DIR = "./out/gridsearch"
os.makedirs(OUT_DIR, exist_ok=True)

ARCH_CONFIGS = [
    ("RRN_golden",             "golden",  False),
    ("RRN_log",                "log",     False),
    ("RRN_two",                "two",     False),
    ("RRN_linear",             "linear",  False),
    ("RRN_golden_norecurrent", "golden",  True),
]

# Parameter grid
LRS = [3e-4, 1e-3, 3e-3]         # Learning rate
WDS = [0.0, 1e-3, 1e-2]          # Rate decay
LSS = [0.0, 0.05]                # Label smoothing

if __name__ == "__main__":
    loaders = {
        s: make_timeseries_loaders(
            batch_size=128,
            val_size=5000,
            seed=s,
            num_workers=NUM_WORKERS,
            print_stats=PRINT_STATS,
        )
        for s in SEEDS
    }

    cols = [
        "arch_name","rhythm","no_recurrence",
        "seed","lr","weight_decay","label_smoothing",
        "best_epoch","best_val_acc",
    ]

    for arch_name, RHYTHM, NO_RECURRENCE in ARCH_CONFIGS:
        print(f"\n===== {arch_name} (rhythm={RHYTHM}, no_recurrence={NO_RECURRENCE}) =====")
        out_csv = os.path.join(OUT_DIR, f"rrn_gridsearch_{arch_name}.csv")

        for lr, wd, ls in itertools.product(LRS, WDS, LSS):
            for seed in SEEDS:
                row = run_one(
                    loaders=loaders,
                    model_fn=lambda RHYTHM=RHYTHM, NO_RECURRENCE=NO_RECURRENCE: RRNModel(
                        rhythm_configuration=RHYTHM,
                        no_recurrence=NO_RECURRENCE,
                    ),
                    prepare_batch=rrn_prepare_batch,
                    arch_name=arch_name,
                    seed=seed,
                    lr=lr,
                    wd=wd,
                    ls=ls,
                    epochs=EPOCHS,
                    extra_row={"rhythm": RHYTHM, "no_recurrence": NO_RECURRENCE},
                )
                append_row(out_csv, cols, row)
                print(arch_name,  "seed", seed, "lr", lr, "wd", wd, "ls", ls, "| val", row["best_val_acc"])
                print("Saved to:", out_csv)


# RRN (fast): Compute and save optimal hyperparameter set for each architecture.

In [ ]:
from Utilities import select_and_save_best_hparams

GRID_DIR   = "./out/gridsearch"
OUT_CSV    = "./out/best_runs/best_hparams_from_grid.csv"
GRID_SEEDS = [123, 456, 789]

best = select_and_save_best_hparams(
    grid_dir=GRID_DIR,
    out_csv=OUT_CSV,
    grid_seeds=GRID_SEEDS,
    pattern="*gridsearch*.csv",
)

print(best)
print("Saved:", OUT_CSV)

# RRN (slow): Run 10 instances of each RRN architecture at the optimal hyperparameters & save metrics (accuracy, confusion matrix)

In [ ]:
import os, csv
import numpy as np
import torch

from load_MNIST    import make_timeseries_loaders
from Utilities     import set_seed, append_row, read_best_params, compute_confusion_matrix
from RRN_functions import RRNModel, rrn_prepare_batch
from train         import TrainConfig, Trainer


# ---------------- Settings ----------------
ARCH_CONFIGS = [
    ("RRN_golden",             "golden",  False),
    ("RRN_log",                "log",     False),
    ("RRN_two",                "two",     False),
    ("RRN_linear",             "linear",  False),
    ("RRN_golden_norecurrent", "golden",  True),
]

BEST_CSV   = "./out/best_runs/best_hparams_from_grid.csv"
OUT_DIR    = "./out/best_runs"
os.makedirs(OUT_DIR, exist_ok=True)

WRES_DIR   = os.path.join(OUT_DIR, "W_res")
os.makedirs(WRES_DIR, exist_ok=True)

CM_DIR     = os.path.join(OUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

OUT_CSV    = os.path.join(OUT_DIR, "rrn_best10runs.csv")

EPOCHS     = 30
EVAL_SEEDS = list(range(101, 111))   # 10 runs per architecture

# keep loaders/training consistent
BATCH_SIZE  = 128
VAL_SIZE    = 5000
NUM_WORKERS = 0
PRINT_STATS = False

if __name__ == "__main__":
    best = read_best_params(BEST_CSV)

    # Start fresh each time
    if os.path.exists(OUT_CSV):
        os.remove(OUT_CSV)

    cols = [
        "arch_name", "rhythm", "no_recurrence",
        "seed",
        "lr", "weight_decay", "label_smoothing",
        "best_epoch", "best_val_acc",
        "test_loss", "test_acc",
    ]

    for arch_name, rhythm, no_recurrence in ARCH_CONFIGS:
        hp = best[arch_name]
        print(f"\n===== {arch_name} (rhythm={rhythm}, no_recurrence={no_recurrence}) =====")
        print("Using best hparams:", hp)

        for seed in EVAL_SEEDS:
            set_seed(seed)

            train_loader, val_loader, test_loader = make_timeseries_loaders(
                batch_size=BATCH_SIZE,
                val_size=VAL_SIZE,
                seed=seed,
                num_workers=NUM_WORKERS,
                print_stats=PRINT_STATS,
            )

            model = RRNModel(rhythm_configuration=rhythm,no_recurrence=no_recurrence)

            cfg = TrainConfig(
                epochs=EPOCHS,
                lr=hp["lr"],
                weight_decay=hp["weight_decay"],
                label_smoothing=hp["label_smoothing"],
                seed=seed,
                history_csv=False,
            )
            os.makedirs(cfg.ckpt_dir, exist_ok=True)
            os.makedirs(cfg.log_dir, exist_ok=True)

            trainer = Trainer(cfg, prepare_batch=rrn_prepare_batch)
            results = trainer.fit(
                model, train_loader, val_loader, test_loader,
                run_name=f"{arch_name}_BEST_seed{seed}"
            )

            # Save W_res if it was estimated (trainable recurrence)
            w_res_path = ""
            if hasattr(model, "W_res") and getattr(model.W_res, "requires_grad", False):
                w_res_path = os.path.join(WRES_DIR, f"W_res_{arch_name}_seed{seed}.npy")
                np.save(w_res_path, model.W_res.detach().cpu().numpy())

            # Compute + save confusion matrix on test set (post-fit; assumes trainer leaves model at best checkpoint for test)
            cm = compute_confusion_matrix(model, test_loader, rrn_prepare_batch, n_classes=10)
            cm_path = os.path.join(CM_DIR, f"confmat_{arch_name}_seed{seed}.npy")
            np.save(cm_path, cm)

            row = dict(
                arch_name=arch_name,
                rhythm=rhythm,
                no_recurrence=no_recurrence,
                seed=seed,
                lr=hp["lr"],
                weight_decay=hp["weight_decay"],
                label_smoothing=hp["label_smoothing"],
                best_epoch=results["best_epoch"],
                best_val_acc=results["best_val_acc"],
                test_loss=results["test"]["loss"],
                test_acc=results["test"]["acc"],
            )

            append_row(OUT_CSV, cols, row)
            print(arch_name, "seed", seed, "| test_acc", row["test_acc"])

    print("\nSaved results:", OUT_CSV)
    print("Saved W_res dir:", WRES_DIR)
    print("Saved confusion matrices dir:", CM_DIR)

# (Figure) Counts and plot of example scanline traces¶

In [ ]:
from keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Load the MNIST data
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(60000, 784).astype(np.float32)
X_train /= 255.0

# Count the labels for training and testing subsets
train_counts = dict(Counter(y_train))
test_counts = dict(Counter(y_test))

# Print the table header
print("{:<10}{:<20}{:<20}".format("Digit", "Training Count", "Testing Count"))
print("-" * 50)

# Print counts for each digit from 0 to 9
for digit in range(10):
    train_count = train_counts.get(digit, 0)
    test_count = test_counts.get(digit, 0)
    print("{:<10},{:<20},{:<20}".format(digit, train_count, test_count))

# Compute totals for training and testing
total_train = sum(train_counts.values())
total_test = sum(test_counts.values())

print("{:<10},{:<20},{:<20}".format("Total", total_train, total_test))

# Indices to plot
indices = [1, 3, 5]

f, axes = plt.subplots(nrows=3, ncols=2, figsize=(6, 3))

for i, k in enumerate(indices):

    # Plot the image
    axes[i, 0].imshow(X_train[k].reshape(28, 28), cmap='gray_r')
    axes[i, 0].axis('off')

    # Plot the scanline
    axes[i, 1].plot(X_train[k], 'k')
    axes[i, 1].set_xlim([0, 784])
    axes[i, 1].set_ylim([0, 1])

    # Remove all axes/spines/ticks
    axes[i, 1].axis('off')

# Add scale bar to the last subplot (bottom-right)
ax = axes[-1, 1]
y_bar = 0.08
ax.plot([0, 100], [y_bar, y_bar], color='k', lw=2, solid_capstyle='butt')
ax.text(50, y_bar + 0.06, '100', ha='center', va='bottom', color='k', fontsize=9)

plt.tight_layout()
f.savefig("./ps/Fig2_A_Scanline_Traces.pdf", bbox_inches='tight')
plt.show()

# (Figure) RRN & LSTM & Standard RNN: Plot the test accuracy

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as spstats

# ---------------- Inputs ----------------
BEST_DIR = "./out/best_runs"

RRN_CSV  = os.path.join(BEST_DIR, "rrn_best10runs.csv")
LSTM_CSV = os.path.join(BEST_DIR, "lstm_best10runs.csv")
RNN_CSV  = os.path.join(BEST_DIR, "rnn_best10runs.csv")

PARAMS_FIXED_BY_KEY = {
    "LSTM_h6": 298,
    "RNN_h11": 296,
    "RRN_golden": 252,
    "RRN_log": 90,
    "RRN_two": 136,
    "RRN_linear": 252,
}

# ---------------- Load ----------------
dfs = []
for p in [RRN_CSV, LSTM_CSV, RNN_CSV]:
    if os.path.exists(p):
        dfs.append(pd.read_csv(p))

df = pd.concat(dfs, ignore_index=True)

# attach params
df["n_params"] = df["arch_name"].map(PARAMS_FIXED_BY_KEY)
df = df.dropna(subset=["n_params"]).copy()
df["n_params"] = df["n_params"].astype(int)

# summarize
g = df.groupby("arch_name", as_index=False).agg(
    n=("test_acc", "count"),
    n_params=("n_params", "first"),
    mean_acc=("test_acc", "mean"),
    sem_acc=("test_acc", lambda x: x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else 0.0),
)
g["label"]  = g["arch_name"]

# Show the dataframe
print(g.to_string(index=False))


# ---- Summary stats per architecture ----
summary_stats = (
    df.groupby("arch_name")["test_acc"]
      .agg(
          n="count",
          mean="mean",
          std=lambda x: x.std(ddof=1) if len(x) > 1 else 0.0
      )
      .reset_index()
)

summary_stats["sem"] = summary_stats.apply(
    lambda r: (r["std"] / np.sqrt(r["n"])) if r["n"] > 1 else 0.0,
    axis=1
)

# Sort by mean test accuracy (descending)
summary_stats = summary_stats.sort_values("mean", ascending=False).reset_index(drop=True)

# x-tick labels should be the PARAMS_FIXED_BY_KEY keys (i.e., architecture names)
labels = summary_stats["arch_name"].tolist()
means  = summary_stats["mean"].to_numpy()
err2   = (2.0 * summary_stats["sem"]).to_numpy()

# ---- Bar plot ----
x = np.arange(len(labels))

plt.figure(figsize=(4, 3.25))
colors = ['gold' if label == 'RRN_golden' else 'gray' for label in labels]
plt.bar(x, means, color=colors, edgecolor="0.7")  # filled gray bars

# 2*SEM as thick black vertical lines (no caps)
plt.errorbar(x, means, yerr=err2, fmt="none", ecolor="k", elinewidth=3, capsize=0)

plt.xticks(x, labels, rotation=30, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.ylabel("Test accuracy")
plt.ylim(0.4, 0.8)
plt.grid(axis='y', alpha=0.35, linestyle='--')
plt.tight_layout()
plt.savefig("./ps/Fig1_B_MNIST_Boxplots.pdf", format="pdf", bbox_inches="tight", pad_inches=0)


# ---- Statistical tests ----

import pandas as pd
import statsmodels.formula.api as smf

GROUPS = ["RRN_golden", "LSTM_h6", "RRN_linear", "RRN_two", "RRN_log", "RNN_h11"]

# Subset to the 5 groups and create an explicit categorical predictor
df_lm          = df[df["arch_name"].isin(GROUPS)].copy()
df_lm["group"] = pd.Categorical(df_lm["arch_name"], categories=GROUPS)

# OLS with intercept + group, using RRN_golden as the reference group
model = smf.ols(
    'test_acc ~ C(group, Treatment(reference="RRN_golden"))',
    data=df_lm
).fit()

out = pd.DataFrame({
    "predictor": model.params.index,
    "coef":      model.params.values,
    "sem":       model.bse.values,
    "p_value":   model.pvalues.values,
})

print(out.to_string(index=False, float_format=lambda x: f"{x:.3g}"))

# #---- Visualize model fits --------------
# resid  = model.resid
# fitted = model.fittedvalues

# # ---- Residuals vs Fitted ----
# plt.figure(figsize=(6.5, 4.5))
# plt.scatter(fitted, resid)
# plt.axhline(0, linewidth=2)
# plt.xlabel("Fitted values")
# plt.ylabel("Residuals")
# plt.title("Residuals vs Fitted")
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

# # ---- Residual histogram ----
# plt.figure(figsize=(6.5, 4.5))
# plt.hist(resid, bins=15)
# plt.axvline(0, linewidth=2)
# plt.xlabel("Residual")
# plt.ylabel("Count")
# plt.title("Residual Histogram")
# plt.grid(True, alpha=0.3)
# plt.tight_layout()
# plt.show()

# # ---- Optional: Q-Q plot (normality check) ----
# import scipy.stats as st

# plt.figure(figsize=(6.5, 4.5))
# st.probplot(resid, dist="norm", plot=plt)
# plt.title("Residual Q-Q Plot")
# plt.tight_layout()
# plt.show()

# (Figure) RRN & LSTM & Standard RNN: Plot the confusion matrices

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from Utilities import row_normalize_confmat, load_confmats_for_arch, draw_confmat_on_ax

CM_DIR  = "./out/best_runs/confusion_matrices"
FIG_DIR = "./out/figs"

ARCH_NAMES = [
    "RRN_golden",
    "LSTM_h6",
    "RNN_h11",
    #"RRN_linear",
    #"RRN_two",
    #"RRN_log",
]

N_CLASSES = 10

# ---------------- Load + average confmats ----------------
avg_cms = {}
for arch_name in ARCH_NAMES:
    cms, paths = load_confmats_for_arch(CM_DIR, arch_name)
    cms_norm = np.stack([row_normalize_confmat(cm) for cm in cms], axis=0)
    avg_cms[arch_name] = cms_norm.mean(axis=0)
    print(f"{arch_name}: loaded {len(paths)} matrices")

# ---------------- Compare diagonals vs RRN_golden ----------------
gold_diag_r = np.round(np.diag(avg_cms["RRN_golden"]), 2)

other_archs = [a for a in ARCH_NAMES if a != "RRN_golden"]

counts = np.zeros(N_CLASSES, dtype=int)
for a in other_archs:
    other_diag_r = np.round(np.diag(avg_cms[a]), 2)
    counts += (gold_diag_r >= other_diag_r).astype(int)

print("\nDiagonal comparison AFTER rounding to 2 decimals:")
print("Count of architectures where RRN_golden > other (per class):")
for i in range(N_CLASSES):
    print(f"  class {i}: {counts[i]} / {len(other_archs)}")

# ---------------- Plot grid ----------------
fig, axes = plt.subplots(1, 3, figsize=(11, 11))
axes = axes.ravel()

for k, arch_name in enumerate(ARCH_NAMES):
    draw_confmat_on_ax(axes[k], avg_cms[arch_name], arch_name, n_classes=N_CLASSES, vmin=0.0, vmax=1.0)

for k in range(len(ARCH_NAMES), len(axes)):
    axes[k].axis("off")

plt.tight_layout()
os.makedirs(FIG_DIR, exist_ok=True)
out_fig = os.path.join(FIG_DIR, "avg_confmat_grid_2x3.pdf")
plt.savefig(out_fig, format="pdf", bbox_inches="tight")

plt.savefig(
    "./ps/Fig2_C_Confusion_Matrices.pdf",
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.02
)
print("\nSaved grid figure:", out_fig)

# (Figure) Plot response of trained network to MNIST digits

In [ ]:
import os
import torch
import random
import numpy as np
from RRN_functions import RRNModel, rrn_extract_features

# Load the model
model = RRNModel(Fs=1000, n_classes=10, rhythm_configuration="golden", verbose=False)

# Location of saved states
device = "cpu"
CKPT_DIR  = "./out/checkpoints"

# Load the testing data
from load_MNIST import load_mnist_numpy_flat01
X_tr_np, y_tr_np, X_te_np, y_te_np = load_mnist_numpy_flat01()

digits = [1,2,3,4]
Kreps  = 100

mn  = np.zeros([model.K, np.size(digits)])
sem = np.zeros([model.K, np.size(digits)])
for [i_digit, digit] in enumerate(digits):

    print('Digit:', digit)

    F_norm = np.zeros((model.K, Kreps), dtype=float)
    model.eval()
    with torch.no_grad():
        for k in range(Kreps):
    
            # Get random RRN model
            seed      = random.randint(101, 200)
            ckpt_path = os.path.join(CKPT_DIR, f"RRN_golden_BEST_seed{seed}_best.pt")
            state     = torch.load(ckpt_path, map_location="cpu")
            model.load_state_dict(state["model_state"])
            
            # Get random digit
            index = random.choice(np.where(y_te_np == digit)[0])
            x     = X_te_np[index]
            
            # Compute features
            f = rrn_extract_features(model, x)
            f = f.unsqueeze(0) 
    
            # ... and store them.
            F_norm[:, k] = f.squeeze(0).cpu().numpy()
    
    mn[:,i_digit]  = np.mean(F_norm,1)
    sd             = np.std( F_norm,1);
    sem[:,i_digit] = sd / np.sqrt(Kreps)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

FIG_DIR = "./out/figs"
fr = np.asarray(np.round(model.frange,1), dtype=float)
plt.figure(figsize=(4, 3.5), dpi=120)
ax = plt.gca()

for j, digit in enumerate(digits):
    m = mn[:, j]
    s = sem[:, j]
    ax.semilogx(model.frange, m, lw=1, label=str(digit))
    ax.fill_between(model.frange, m - 1.96*s, m + 1.96*s, alpha=0.18)

# --- Set ticks  ---
ax.set_xticks(fr)
ax.set_xticklabels([f'{x:.1f}' for x in fr], rotation=90)
ax.set_yscale("log")
ax.xaxis.set_minor_locator(mticker.NullLocator())
ax.yaxis.set_minor_locator(mticker.NullLocator())  # remove minor y ticks (optional but clean)

# --- Label axes ---
ax.set_xlabel("Frequency (Hz)", fontsize=12)
ax.set_ylabel("Accumulated Amplitude Squared (a.u.)", fontsize=12)
ax.grid(True, which="both", alpha=0.25)
ax.legend(title="Digit", frameon=True, ncol=1, loc="best")
ax.set_xlim(min(model.frange), max(model.frange))
ax.tick_params(axis='x', labelsize=10)
ax.tick_params(axis='y', labelsize=10)
plt.tight_layout()

plt.savefig("./ps/Fig3_MNIST_Responses.pdf", format="pdf", bbox_inches="tight", pad_inches=0.01)

# RRN (slow): Run more instances of the golden RRN and save estimated connectivity matrix.

In [ ]:
import os, csv
import numpy as np

from load_MNIST    import make_timeseries_loaders
from Utilities     import set_seed, append_row, read_best_params
from RRN_functions import RRNModel, rrn_prepare_batch
from train         import TrainConfig, Trainer


# ---------------- Settings ----------------
ARCH_CONFIGS = [
    ("RRN_golden",             "golden",  False),
]

BEST_CSV   = "./out/best_runs/best_hparams_from_grid.csv"
OUT_DIR    = "./out/best_runs"
os.makedirs(OUT_DIR, exist_ok=True)

WRES_DIR   = os.path.join(OUT_DIR, "W_res")
os.makedirs(WRES_DIR, exist_ok=True)

OUT_CSV    = os.path.join(OUT_DIR, "rrn_best10runs.csv")

EPOCHS     = 30
EVAL_SEEDS = list(range(100,201))   # 100 more runs

# keep loaders/training consistent
BATCH_SIZE = 128
VAL_SIZE   = 5000
NUM_WORKERS = 0
PRINT_STATS = False

if __name__ == "__main__":
    best = read_best_params(BEST_CSV)

    # Start fresh each time
    if os.path.exists(OUT_CSV):
        os.remove(OUT_CSV)

    cols = [
        "arch_name", "rhythm", "no_recurrence",
        "seed",
        "lr", "weight_decay", "label_smoothing",
        "best_epoch", "best_val_acc",
        "test_loss", "test_acc",
        "w_res_path",
    ]

    for arch_name, rhythm, no_recurrence in ARCH_CONFIGS:
        hp = best[arch_name]
        print(f"\n===== {arch_name} (rhythm={rhythm}, no_recurrence={no_recurrence}) =====")
        print("Using best hparams:", hp)

        for seed in EVAL_SEEDS:
            set_seed(seed)

            train_loader, val_loader, test_loader = make_timeseries_loaders(
                batch_size=BATCH_SIZE,
                val_size=VAL_SIZE,
                seed=seed,
                num_workers=NUM_WORKERS,
                print_stats=PRINT_STATS,
            )

            rrn = RRNModel(
                rhythm_configuration=rhythm,
                no_recurrence=no_recurrence,
            )  # uses default init_scale=0.3, sigma=0.0 unless your class defaults changed

            cfg = TrainConfig(
                epochs=EPOCHS,
                lr=hp["lr"],
                weight_decay=hp["weight_decay"],
                label_smoothing=hp["label_smoothing"],
                seed=seed,
                history_csv=False,
            )
            os.makedirs(cfg.ckpt_dir, exist_ok=True)
            os.makedirs(cfg.log_dir, exist_ok=True)

            trainer = Trainer(cfg, prepare_batch=rrn_prepare_batch)
            results = trainer.fit(rrn, train_loader, val_loader, test_loader,
                                  run_name=f"{arch_name}_BEST_seed{seed}")

            # Save W_res if it was estimated (trainable recurrence)
            w_res_path = ""
            if hasattr(rrn, "W_res") and getattr(rrn.W_res, "requires_grad", False):
                w_res_path = os.path.join(WRES_DIR, f"W_res_{arch_name}_seed{seed}.npy")
                np.save(w_res_path, rrn.W_res.detach().cpu().numpy())

            row = dict(
                arch_name=arch_name,
                rhythm=rhythm,
                no_recurrence=no_recurrence,
                seed=seed,
                lr=hp["lr"],
                weight_decay=hp["weight_decay"],
                label_smoothing=hp["label_smoothing"],
                best_epoch=results["best_epoch"],
                best_val_acc=results["best_val_acc"],
                test_loss=results["test"]["loss"],
                test_acc=results["test"]["acc"],
                w_res_path=w_res_path,
            )

            append_row(OUT_CSV, cols, row)
            print(arch_name, "seed", seed, "| test_acc", row["test_acc"], "| w_res", ("YES" if w_res_path else "NO"))

    print("\nSaved results:", OUT_CSV)
    print("Saved W_res dir:", WRES_DIR)


# (Figure): Plot the estimated connectivity matrix

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from RRN_functions import RRNModel
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle

# ---- Load freqs (for tick labels) ----
rrn_tmp = RRNModel(rhythm_configuration="golden")
freqs   = np.asarray(rrn_tmp.frange)

# ---- Load W_res files ----
files = sorted(glob.glob("./out/best_runs/W_res/W_res_RRN_golden_seed*.npy"))
print("Found", len(files), "files")

Ws = np.stack([np.load(p) for p in files], axis=0)  # (N, K, K)
N, K, _ = Ws.shape

# ---- Summary matrices ----
W_mean = Ws.mean(axis=0)

# ---- Separate color scales ----
m_top  = np.max(np.abs(Ws))
m_mean = np.max(np.abs(W_mean))

norm_top  = TwoSlopeNorm(vmin=-m_top,  vcenter=0.0, vmax=m_top)
norm_mean = TwoSlopeNorm(vmin=-m_mean, vcenter=0.0, vmax=m_mean)

# ---- Frequency bands used to analyze weight distributions ----
low_pre_idx   = np.where(freqs <= 9)[0]
high_pre_idx  = np.where((freqs >= 20) & (freqs <= 100))[0]
low_post_idx  = np.where(freqs <= 9)[0]
high_post_idx = np.where((freqs >= 20) & (freqs <= 100))[0]

if len(low_pre_idx) == 0 or len(high_pre_idx) == 0:
    raise ValueError("No frequencies found in one of the requested bands.")

# ---- Compute per-run block means for histogram ----
low_to_high = []
high_to_low = []

for W in Ws:
    low_to_high_block = W[np.ix_(high_post_idx, low_pre_idx)]
    high_to_low_block = W[np.ix_(low_post_idx, high_pre_idx)]
    low_to_high.append(np.mean(low_to_high_block))
    high_to_low.append(np.mean(high_to_low_block))

low_to_high = np.array(low_to_high)
high_to_low = np.array(high_to_low)

t_stat, p_val = stats.ttest_ind(low_to_high, high_to_low)
print("ttest2 p-value:", p_val)
print("ttest2 stats:",   t_stat)

# ---- Choose 4 example runs ----
example_idxs = np.arange(min(4, N))

# ---- Create layout ----
fig = plt.figure(figsize=(14, 9))
gs = fig.add_gridspec(2, 4, height_ratios=[1, 2], wspace=0.45, hspace=0.45)

# First row: 4 example matrices
top_axes = [fig.add_subplot(gs[0, i]) for i in range(4)]

# Second row: each plot spans 2 columns
ax_mean = fig.add_subplot(gs[1, 0:2])
ax_hist = fig.add_subplot(gs[1, 2:4])

# =========================
# First row: 4 example matrices
# =========================
im_top = None
for col, ax in enumerate(top_axes):
    if col < len(example_idxs):
        idx = example_idxs[col]
        W = Ws[idx]
        im_top = ax.imshow(W, aspect="equal", origin="lower", cmap="RdBu_r", norm=norm_top)

        idxs = np.arange(K)
        ax.set_xticks(idxs)
        ax.set_yticks(idxs)
        ax.set_xticklabels([f"{freqs[i]:.1f}" for i in idxs], rotation=90, fontsize=10)
        ax.set_yticklabels([f"{freqs[i]:.1f}" for i in idxs], fontsize=10)

        ax.set_xlabel("Pre frequency (Hz)", fontsize=12)
        if col == 0:
            ax.set_ylabel("Post frequency (Hz)", fontsize=12)
        else:
            ax.set_ylabel("")

        ax.set_title(f"Example run {idx+1}", fontsize=12)
    else:
        ax.axis("off")

# =========================
# Second row, cols 1-2: mean matrix with boxes
# =========================
im_mean = ax_mean.imshow(W_mean, aspect="equal", origin="lower", cmap="RdBu_r", norm=norm_mean)

idxs = np.arange(K)
ax_mean.set_xticks(idxs)
ax_mean.set_yticks(idxs)
ax_mean.set_xticklabels([f"{freqs[i]:.1f}" for i in idxs], rotation=90, fontsize=10)
ax_mean.set_yticklabels([f"{freqs[i]:.1f}" for i in idxs], fontsize=10)

ax_mean.set_xlabel("Pre frequency (Hz)", fontsize=12)
ax_mean.set_ylabel("Post frequency (Hz)", fontsize=12)
ax_mean.set_title("Mean connectivity", fontsize=12)

# Box for low-to-high block
x0 = low_pre_idx[0] - 0.5
y0 = high_post_idx[0] - 0.5
w  = low_pre_idx[-1] - low_pre_idx[0] + 1
h  = high_post_idx[-1] - high_post_idx[0] + 1
rect1 = Rectangle((x0, y0), w, h, fill=False, edgecolor='black', linewidth=2)
ax_mean.add_patch(rect1)

# Box for high-to-low block
x0 = high_pre_idx[0] - 0.5
y0 = low_post_idx[0] - 0.5
w  = high_pre_idx[-1] - high_pre_idx[0] + 1
h  = low_post_idx[-1] - low_post_idx[0] + 1
rect2 = Rectangle((x0, y0), w, h, fill=False, edgecolor='black', linewidth=2)
ax_mean.add_patch(rect2)

# =========================
# Second row, cols 3-4: histogram
# =========================
bins = np.arange(-0.3, 0.45, 0.05)

ax_hist.hist(high_to_low, bins=bins, alpha=0.7, label="high-to-low", edgecolor='black')
ax_hist.hist(low_to_high, bins=bins, alpha=0.7, label="low-to-high", edgecolor='black')

ax_hist.set_xlabel("Mean weight per run", fontsize=12)
ax_hist.set_ylabel("Count", fontsize=12)
ax_hist.set_title("Block-average weight distribution", fontsize=12)
ax_hist.legend(fontsize=10)
ax_hist.grid(axis="y", alpha=0.5)

# ---- Separate colorbars ----
# top-row colorbar
cax_top = fig.add_axes([0.92, 0.61, 0.015, 0.22])
fig.colorbar(im_top, cax=cax_top)

# bottom mean-matrix colorbar
cax_mean = fig.add_axes([0.47, 0.12, 0.015, 0.34])
fig.colorbar(im_mean, cax=cax_mean)

plt.savefig("./ps/Fig4_MNIST_Connectivity.pdf", format="pdf", bbox_inches="tight", pad_inches=0.01)